# Notebook 3: Validate Exported Adapter Directly

Bu notebook tek bir adapter'i dogrudan yukler, metadata bilgisini okur, tek goruntu tahmini yapar ve isterseniz bir klasor uzerinde hizli saglik kontrolu calistirir.
Tek goruntu tahmini artik ayni dosyadan uretilen deterministik derived-view sonuclarini, opsiyonel occlusion sensitivity haritasini veya DINOv3 ViT attention map ciktisini da gosterebilir.

Notebook proje klasoru ve yaygin Drive koklerinde adapter bundle arar. Elle yol vermek de desteklenir.

Manuel giris alanlari:
- `ADAPTER_DIR`: adapter klasoru, export klasoru, telemetry run klasoru veya `adapter_meta.json`
- `ADAPTER_ROOT`: `models/adapters/` gibi crop klasorlerinin ebeveyni
- `IMAGE_PATH`: tek bir goruntu
- `BATCH_IMAGE_DIR`: bir klasor dolusu goruntu

Ayni oturum icinde tekrar denemek icin:
- `FORCE_ADAPTER_RESCAN = True` yapip adapter kesif hucresini tekrar calistirin
- tek goruntu hucresi varsayilan olarak yeni upload ister
- mevcut upload'i korumak icin `REUSE_EXISTING_IMAGE_PATH = True` yapin
- hemen yeni upload zorlamak icin `FORCE_IMAGE_UPLOAD = True` yapin


## Hizli Akis

Bu notebook routeri atlayip secilen adapter bundle uzerinde dogrudan smoke test ve tek goruntu tahmini yapar.

- Adapter otomatik bulunmuyorsa `ADAPTER_DIR` verin; bulunuyorsa listeden `SELECTED_ADAPTER_INDEX` secin.
- Yeni goruntu icin `FORCE_IMAGE_UPLOAD = True`, ayni goruntuyu korumak icin `REUSE_EXISTING_IMAGE_PATH = True` kullanin.
- Skorlar hizli saglik kontroludur; kalibre accuracy olasiligi gibi yorumlamayin.
- Daha basit butonlu deneme icin Notebook 4 kullanin.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if CLONE_TARGET.exists():
        shutil.rmtree(CLONE_TARGET)
    print(f'[BOOTSTRAP] Cloning repo to {CLONE_TARGET}...', flush=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell01_bootstrap_access.py', globals())

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell02_access_notice.py', globals())


In [ ]:
import json
from pathlib import Path

from IPython.display import display, HTML

# Hizli kullanim: Adapter otomatik bulunuyorsa SELECTED_ADAPTER_INDEX secin; yoksa ADAPTER_DIR verin.
# IMAGE_PATH=None Colab'da upload akisini kullanir.
CROP_NAME = None
# Example: 'tomato'. Leave None to list all adapters.

ADAPTER_DIR = None
# Accepted examples:
# - .../continual_sd_lora_adapter/
# - .../artifacts/adapter_export/
# - outputs/colab_notebook_training/
# - .../telemetry/<RUN_ID>/ or .../telemetry/<RUN_ID>/artifacts/
# - .../adapter_meta.json

ADAPTER_ROOT = None
# Usually the parent of crop folders like models/adapters/

INCLUDE_RUN_ADAPTERS = False
# Set True only when you intentionally want to inspect historical runs/ artifacts.

SEARCH_ROOTS = [
    ROOT / 'outputs' / 'colab_notebook_training',
    ROOT / 'outputs' / 'colab_notebook_training' / 'telemetry_runtime' / 'telemetry',
    ROOT / 'models' / 'adapters',
]
if INCLUDE_RUN_ADAPTERS:
    SEARCH_ROOTS.append(ROOT / 'runs')
SELECTED_ADAPTER_INDEX = 0
FORCE_ADAPTER_RESCAN = False
IMAGE_PATH = None
FORCE_IMAGE_UPLOAD = False
# Single image file
BATCH_IMAGE_DIR = None
# Optional folder for batch testing
DEVICE = 'cuda'
CONFIG_ENV = 'colab'
REUSE_EXISTING_IMAGE_PATH = False
ENABLE_ROBUST_SMOKE = True
ROBUST_VIEWS = ('full_resize', 'resize_pad', 'center_crop')
ENABLE_PREDICTION_VISUALIZATION = True
EXPLANATION_METHOD = 'attention_map'
# Options: 'attention_map' or 'occlusion_sensitivity'
EXPLANATION_GRID_SIZE = 7

print(f'[AYAR] device={DEVICE} config_env={CONFIG_ENV}')
print('[ADAPTER] ADAPTER_DIR doluysa kesif atlanir; bos ise SEARCH_ROOTS altinda adapter aranir.')
print('[ADAPTER] Aranacak kokler:')
for root in SEARCH_ROOTS:
    print(f'- {root}')
print('[YOL] ADAPTER_ROOT crop klasorlerinin ebeveyni, IMAGE_PATH tek dosya, BATCH_IMAGE_DIR klasor olmalidir.')
print('[YENILE] Adapter listesini bastan kurmak icin FORCE_ADAPTER_RESCAN=True yapin.')
print('[GORUNTU] Tek goruntu hucresi varsayilan olarak yeni upload ister; mevcut gorseli korumak icin REUSE_EXISTING_IMAGE_PATH=True.')
print(f'[SMOKE] robust={ENABLE_ROBUST_SMOKE} views={ROBUST_VIEWS}')
print(f'[EXPLAIN] enabled={ENABLE_PREDICTION_VISUALIZATION} method={EXPLANATION_METHOD} grid={EXPLANATION_GRID_SIZE}x{EXPLANATION_GRID_SIZE}')
print('[SONRAKI] Adapter kesif hucresini, sonra adapter summary ve tek goruntu hucresini calistirin.')


In [ ]:
# Defer adapter discovery imports until now (when actually needed)
try:
    import ipywidgets as widgets
except Exception:
    widgets = None

from scripts.colab_adapter_smoke_test import discover_adapter_candidates

adapter_candidates = []
adapter_selector = None

if FORCE_ADAPTER_RESCAN:
    ADAPTER_DIR = None
    print('FORCE_ADAPTER_RESCAN=True, rebuilding adapter list.')

if ADAPTER_DIR is not None:
    SELECTED_ADAPTER_DIR = str(Path(ADAPTER_DIR))
    print(f'Using provided ADAPTER_DIR: {SELECTED_ADAPTER_DIR}')
else:
    adapter_candidates = discover_adapter_candidates(SEARCH_ROOTS, crop_name=CROP_NAME)
    if not adapter_candidates:
        raise FileNotFoundError(
            'No adapters found under SEARCH_ROOTS. If needed, provide ADAPTER_DIR manually or update SEARCH_ROOTS.'
        )

    print('Found adapters:')
    for index, candidate in enumerate(adapter_candidates):
        print(f'[{index}] {candidate["display_name"]}')

    if widgets is not None:
        adapter_selector = widgets.Dropdown(
            options=[(candidate['display_name'], index) for index, candidate in enumerate(adapter_candidates)],
            value=min(SELECTED_ADAPTER_INDEX, len(adapter_candidates) - 1),
            description='Adapter:',
            layout=widgets.Layout(width='95%'),
        )
        display(adapter_selector)
        print('Select adapter from dropdown, then run next cell.')
    else:
        print('ipywidgets not available. Set SELECTED_ADAPTER_INDEX to your desired adapter and run next cell.')

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell05_adapter_summary.py', globals())


In [ ]:
# Defer prediction and visualization imports until needed (now)
try:
    import pandas as pd
except Exception:
    pd = None

from scripts.colab_adapter_smoke_test import (
    build_prediction_visualization_images,
    predict_single_image,
)

request_new_upload = running_in_colab() and not bool(REUSE_EXISTING_IMAGE_PATH)
adapter_summary = globals().get('summary')
if (ADAPTER_DIR is None or CROP_NAME is None) and isinstance(adapter_summary, dict):
    ADAPTER_DIR = adapter_summary.get('resolved_adapter_dir', ADAPTER_DIR)
    CROP_NAME = adapter_summary.get('crop_name', CROP_NAME)

if ADAPTER_DIR is None and CROP_NAME is None:
    raise RuntimeError(
        'Run adapter selection/summary cell first so Notebook 3 can resolve ADAPTER_DIR and CROP_NAME.'
    )

if FORCE_IMAGE_UPLOAD:
    IMAGE_PATH = None
    print('FORCE_IMAGE_UPLOAD=True, requesting new image upload.')
elif request_new_upload:
    IMAGE_PATH = None
    print('REUSE_EXISTING_IMAGE_PATH=False, new image upload requested for this run.')

if IMAGE_PATH is None:
    if not running_in_colab():
        raise ValueError('Running outside Colab: set IMAGE_PATH to a single image file.')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Upload cancelled or no file selected.')
    IMAGE_PATH = next(iter(uploaded.keys()))
    FORCE_IMAGE_UPLOAD = False

IMAGE_PATH = str(Path(IMAGE_PATH))
print(f'Using IMAGE_PATH: {IMAGE_PATH}')

single_result = predict_single_image(
    IMAGE_PATH,
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env=CONFIG_ENV,
    device=DEVICE,
    enable_robust_smoke=ENABLE_ROBUST_SMOKE,
    robust_views=ROBUST_VIEWS,
    explain_prediction=ENABLE_PREDICTION_VISUALIZATION,
    explanation_grid_size=EXPLANATION_GRID_SIZE,
    explanation_method=EXPLANATION_METHOD,
)

view_consistency = dict(single_result.get('view_consistency', {}))
uncertainty = dict(single_result.get('uncertainty_diagnostics', {}))

print('Notebook 3 note: class confidence is top-1 softmax score, not calibrated accuracy probability.')
print('OOD decision and class confidence answer different questions; OOD summarizes distribution fit, confidence summarizes class preference.')

summary_rows = [
    {'metric': 'status', 'value': single_result.get('status')},
    {'metric': 'error', 'value': single_result.get('error')},
    {'metric': 'predicted_class', 'value': single_result.get('predicted_class')},
    {'metric': 'confidence', 'value': None if single_result.get('status') == 'error' else round(float(single_result.get('confidence', 0.0)), 4)},
    {'metric': 'is_ood', 'value': single_result.get('is_ood')},
    {'metric': 'score_method', 'value': single_result.get('score_method')},
    {'metric': 'primary_score', 'value': single_result.get('primary_score')},
    {'metric': 'decision_threshold', 'value': single_result.get('decision_threshold')},
    {'metric': 'primary_view', 'value': view_consistency.get('primary_view')},
    {'metric': 'stable_across_views', 'value': view_consistency.get('stable')},
]
if pd is not None:
    display(pd.DataFrame(summary_rows))
else:
    print(summary_rows)

view_rows = list(single_result.get('views', []))
if view_rows:
    print('Derived view results:')
    if pd is not None:
        display(pd.DataFrame(view_rows))
    else:
        print(view_rows)

visualization_images = build_prediction_visualization_images(IMAGE_PATH, single_result)
if visualization_images:
    print('Model view and explanation heatmap. Attention map shows attention routing; occlusion sensitivity shows confidence drop when bright regions are masked.')
    display(visualization_images['model_view'])
    display(visualization_images['heatmap_overlay'])

print('Prediction status:', single_result.get('status'))
if single_result.get('error'):
    print('Prediction error:', single_result.get('error'))
print('View warnings:', view_consistency.get('warning_codes', []))
print('Uncertainty warnings:', uncertainty.get('warning_codes', []))
display(
    HTML(
        '<details><summary>Raw JSON</summary><pre>'
        + json.dumps(single_result, indent=2)
        + '</pre></details>'
    )
)

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell07_batch_prediction.py', globals())
